# Binary Logistic Regression — Glass Dataset

We implement logistic regression from scratch using NumPy. The Glass Identification dataset contains 9 chemical properties. We create a binary problem: Type 1 glass vs everything else.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("glass.csv")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head().to_string())

Dataset shape: (214, 10)
Columns: ['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
         RI         Na        Mg        Al         Si         K         Ca        Ba        Fe  Type
0  1.519614  14.985255  1.579118  0.976634  69.219298  1.777849   8.591074  1.139357  0.390381     3
1  1.532866  11.660744  0.526802  1.347992  71.832402  1.401647  10.694828  2.590001  0.229900     2
2  1.527836  12.277798  0.643463  0.683239  72.389047  0.252559   5.966531  1.557245  0.029082     3
3  1.524769  15.225440  3.426798  3.149687  69.394252  0.107244   8.856896  1.437546  0.497433     1
4  1.514588  14.547430  2.781981  2.199496  73.653166  5.926334   5.365234  0.076926  0.028890     2


## Constructing Binary Labels

Samples of Type 1 get label 1, all others get label 0.

In [1]:
df["binary_label"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])
X_data = df.drop(columns=["binary_label"]).values
y_data = df["binary_label"].values
print("Positive class (Type 1):", y_data.sum(), "/ Total:", len(y_data))

Positive class (Type 1): 69 / Total: 214


## Splitting and Scaling

An 80/20 train-test split is used. StandardScaler normalizes the 9 chemical features.

In [1]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42
)
norm = StandardScaler()
X_tr = norm.fit_transform(X_tr)
X_te = norm.transform(X_te)
print("Train samples:", X_tr.shape)
print("Test  samples:", X_te.shape)

Train samples: (171, 9)
Test  samples: (43, 9)


## Sigmoid Function

The sigmoid maps any real-valued number into (0, 1):

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [1]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def predict_proba(X, W, b):
    linear_out = X @ W + b
    return sigmoid(linear_out)

print("sigmoid(0)  =", sigmoid(0))
print("sigmoid(2)  =", round(sigmoid(2), 4))
print("sigmoid(-2) =", round(sigmoid(-2), 4))

sigmoid(0)  = 0.5
sigmoid(2)  = 0.8808
sigmoid(-2) = 0.1192


## Loss and Gradient Step

Binary cross-entropy is used as the loss. The gradient simplifies to $(p - y)$.

In [1]:
def cross_entropy_loss(y_true, y_prob):
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

def update_weights(X, y_true, W, b, learning_rate):
    y_prob = predict_proba(X, W, b)
    error  = y_prob - y_true
    W = W - learning_rate * (X.T @ error) / len(y_true)
    b = b - learning_rate * np.mean(error)
    return W, b

## Training the Model

Parameters start at zero. Each epoch does a full pass of gradient descent.

In [1]:
W = np.zeros(X_tr.shape[1])
b = 0.0
lr = 0.1
num_epochs = 100

for ep in range(num_epochs):
    W, b = update_weights(X_tr, y_tr, W, b, lr)
    if ep % 20 == 0:
        loss = cross_entropy_loss(y_tr, predict_proba(X_tr, W, b))
        print(f"Epoch {ep:3d}  |  Loss: {loss:.4f}")

Epoch   0  |  Loss: 0.6894
Epoch  20  |  Loss: 0.6446
Epoch  40  |  Loss: 0.6282
Epoch  60  |  Loss: 0.6218
Epoch  80  |  Loss: 0.6190


## Applying Decision Thresholds

Using 0.7 means the model only predicts class 1 when highly confident — fewer false positives.

In [1]:
def classify(probs, cutoff=0.5):
    return (probs >= cutoff).astype(int)

prob_test = predict_proba(X_te, W, b)

y_hat_50 = classify(prob_test, 0.5)
y_hat_70 = classify(prob_test, 0.7)

print("Predictions at threshold 0.5:", y_hat_50)
print("Predictions at threshold 0.7:", y_hat_70)
print("Accuracy at 0.5:", round(np.mean(y_hat_50 == y_te), 3))
print("Accuracy at 0.7:", round(np.mean(y_hat_70 == y_te), 3))

Predictions at threshold 0.5: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0]
Predictions at threshold 0.7: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0]
Accuracy at 0.5: 0.721
Accuracy at 0.7: 0.721
